# Korean Specialization 목록 튜토리얼

`korean_specialization.yaml`은 한국어 모델 전용 평가를 위한 seed group 정의 파일입니다.

## 2개 Seed Group 요약

| Group | 목적 | seed 수 | 비용 | 예상 시간 |
|---|---|---:|---|---|
| `priority_ko_soft_20m` | 중요 리스크 우선 점검 (jailbreak, 인젝션, 유해성, 악성코드) | 7 | medium | ~10분 (cap=3 기준) |
| `quick_variety_smoke_ko` | 다양한 공격 유형 빠른 스모크 테스트 | 9 | low | 빠름 |

## 공통 설정
- `target_lang: ko` 고정
- `generations: 1` (샘플당 응답 1회)
- `soft_seed_prompt_cap`: CLI `--soft_seed_prompt_cap` 인자로 지정 (미지정 시 기본값 256)

## 사용 예시
```bash
python -m garak --target_type openai --target_name gpt-4o-mini \
  --seed_groups_file src/garak/configs/korean_specialization.yaml \
  --seed_group priority_ko_soft_20m \
  --soft_seed_prompt_cap 3
```

## 1) Seed Group 한눈에 보기

아래 코드는 `korean_specialization.yaml`을 로드하여 group별 요약표와 상세 seed 목록을 출력합니다.

In [4]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", None)

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "main.py").exists() else cwd.parent

cfg_path = repo_root / "src" / "garak" / "configs" / "korean_specialization.yaml"
assert cfg_path.exists(), f"파일이 없습니다: {cfg_path}"

raw = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
groups = raw.get("seed_groups", []) if isinstance(raw, dict) else (raw if isinstance(raw, list) else [])
assert groups, f"seed_groups가 비어 있습니다. (type={type(raw).__name__})"

print(f"loaded: {cfg_path}")
print(f"groups: {len(groups)}\n")
for i, g in enumerate(groups, start=1):
    print(f"  {i}. {g.get('id', '(no id)')}")

loaded: /Users/selectstar/garak_ko/src/garak/configs/korean_specialization.yaml
groups: 2

  1. priority_ko_soft_20m
  2. quick_variety_smoke_ko


In [6]:
def fmt_seed_list(seed_names, limit=120):
    shown = seed_names[:limit]
    extra = len(seed_names) - len(shown)
    body = "<br>".join(f"`{s}`" for s in shown)
    if extra > 0:
        body += f"<br>… (+{extra} more)"
    return body or "(none)"

def fmt_tags(tags):
    if not tags:
        return "-"
    parts = []
    for t in tags:
        if isinstance(t, dict):
            parts.extend(f"{k}: {v}" for k, v in t.items())
        else:
            parts.append(str(t))
    return ", ".join(parts)

summary_rows = []
details_blocks = []

for g in groups:
    gid = g.get("id", "(no id)")
    desc = g.get("description", "-")
    tags = g.get("tags", [])
    run = g.get("run", {}) or {}
    seeds = run.get("seeds", []) or []
    seed_names = [s.get("seed", "") for s in seeds if s.get("seed")]

    # seed family 추출
    families = list(dict.fromkeys(
        s.split(".", 1)[0] if "." in s else s for s in seed_names if s
    ))

    summary_rows.append({
        "group_id": gid,
        "name": g.get("name", ""),
        "기능 설명": desc,
        "tags": fmt_tags(tags),
        "run.generations": run.get("generations", "(inherit)"),
        "run.soft_seed_prompt_cap": run.get("soft_seed_prompt_cap", "(inherit: config)"),
        "seed_count": len(seed_names),
        "seed_families": ", ".join(families) if families else "-",
    })

    details_blocks.append("\n".join([
        f"### `{gid}`",
        f"- 기능 설명: {desc}",
        f"- tags: `{fmt_tags(tags)}`",
        f"- target_lang: `{run.get('target_lang', '(inherit)')}`",
        f"- generations: `{run.get('generations', '(inherit)')}`",
        f"- soft_seed_prompt_cap: `{run.get('soft_seed_prompt_cap', '(inherit: config)')}`",
        f"- seeds ({len(seed_names)}개): `{', '.join(seed_names)}`" if seed_names else "- seeds: (empty)",
        "",
        "<details><summary>seed 목록 + 설명 펼치기</summary>",
        "",
        fmt_seed_list(seed_names, limit=150),
        "",
        "</details>",
    ]))

display(Markdown(f"### Korean Specialization Group 요약표\nconfig: `{cfg_path}`"))

df_summary = pd.DataFrame(summary_rows).sort_values(
    ["seed_count", "group_id"], ascending=[False, True]
).reset_index(drop=True)
display(df_summary)

display(Markdown("### Group별 상세 보기"))
display(Markdown("\n\n---\n\n".join(details_blocks)))

### Korean Specialization Group 요약표
config: `/Users/selectstar/garak_ko/src/garak/configs/korean_specialization.yaml`

,group_id,name,기능 설명,tags,run.generations,run.soft_seed_prompt_cap,seed_count,seed_families
0,quick_variety_smoke_ko,Quick variety smoke (ko),간단 스모크: dan/grandma/encoding/continuation/phrasing/divergence/snowball/ansiescape/doctor,"run.stage: smoke, lang.coverage: ko, cost: low",1,(inherit: config),9,"dan, grandma, encoding, continuation, phrasing, divergence, snowball, ansiescape, doctor"
1,priority_ko_soft_20m,"Priority package (ko, run-soft, ~20m)","중요 기능 우선: jailbreak(tap/suffix), 잠복·프롬프트 인젝션, 유해성, 취약점 악용, 허위정보, 악성코드","run.stage: prioritized, profile: run-soft, lang.coverage: ko, cost: medium",1,(inherit: config),7,"tap, suffix, latentinjection, promptinject, atkgen, lmrc, malwaregen"


### Group별 상세 보기

### `priority_ko_soft_20m`
- 기능 설명: 중요 기능 우선: jailbreak(tap/suffix), 잠복·프롬프트 인젝션, 유해성, 취약점 악용, 허위정보, 악성코드
- tags: `run.stage: prioritized, profile: run-soft, lang.coverage: ko, cost: medium`
- target_lang: `ko`
- generations: `1`
- soft_seed_prompt_cap: `(inherit: config)`
- seeds (7개): `tap.TAPCached, suffix.GCGCached, latentinjection.LatentInjectionReport, promptinject.HijackLongPrompt, atkgen.Tox, lmrc.SlurUsage, malwaregen.Payload`

<details><summary>seed 목록 + 설명 펼치기</summary>

`tap.TAPCached`<br>`suffix.GCGCached`<br>`latentinjection.LatentInjectionReport`<br>`promptinject.HijackLongPrompt`<br>`atkgen.Tox`<br>`lmrc.SlurUsage`<br>`malwaregen.Payload`

</details>

---

### `quick_variety_smoke_ko`
- 기능 설명: 간단 스모크: dan/grandma/encoding/continuation/phrasing/divergence/snowball/ansiescape/doctor
- tags: `run.stage: smoke, lang.coverage: ko, cost: low`
- target_lang: `ko`
- generations: `1`
- soft_seed_prompt_cap: `(inherit: config)`
- seeds (9개): `dan, grandma, encoding, continuation, phrasing, divergence, snowball, ansiescape, doctor`

<details><summary>seed 목록 + 설명 펼치기</summary>

`dan`<br>`grandma`<br>`encoding`<br>`continuation`<br>`phrasing`<br>`divergence`<br>`snowball`<br>`ansiescape`<br>`doctor`

</details>